# Production

Opens the production page, where one button produces a language from start to
finish: scene descriptions, recordings, pictures, study units, web content,
builds, and publishing.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

When the last cell prints a link, open it and sign in with the user name and
password it shows. Keep this tab open while the page is working.

Everything is written to Google Drive, in `My Drive/language_system`. If the
session disconnects, nothing is lost: run all again and press Start; finished
work is skipped.

In [ ]:
#@title 1 · Settings
LANGUAGE_MODEL = True        #@param {type:"boolean"}
# True: install the local language model for scene descriptions (about 5 GB, a few minutes)
FRENCH_VOICE = False         #@param {type:"boolean"}
# True only when producing French (Kokoro voice). German needs nothing extra.

In [ ]:
#@title 2 · Connect Google Drive and install or update the code
import os, shutil, zipfile, pathlib, subprocess
from google.colab import drive
drive.mount('/content/drive')

HOME = pathlib.Path('/content/drive/MyDrive/language_system')
HOME.mkdir(parents=True, exist_ok=True)

# The code arrives as language_system_production.zip in My Drive (top level)
# or in My Drive/language_system. Code folders are replaced; datasets are only
# added when missing, never overwritten; generated material is never touched.
CODE = ('bin/', 'languages/', 'app/', 'notebooks/', 'docs/', 'README.md', 'requirements.txt')

def produced(csv_path):
    import csv
    with open(csv_path, encoding='utf-8') as f:
        return any((r.get('audio_ref') or '').strip() for r in csv.DictReader(f))
for place in (HOME.parent, HOME):
    z = place / 'language_system_production.zip'
    if z.exists():
        with zipfile.ZipFile(z) as zf:
            for name in zf.namelist():
                rel = name.split('/', 1)[1] if name.startswith('language_system/') else name
                if not rel or rel.endswith('/'):
                    continue
                dest = HOME / rel
                if rel.startswith(CODE):
                    pass
                elif rel.startswith('data/') and (not dest.exists() or not produced(dest)):
                    pass          # a dataset is updated only while no recordings are linked to it
                else:
                    continue
                dest.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(name) as src, open(dest, 'wb') as out:
                    shutil.copyfileobj(src, out)
        print('code installed from', z)
        break
else:
    print('no zip found; using the code already in', HOME)

assert (HOME / 'bin' / 'produce.py').exists(), 'bin/produce.py missing: put language_system_production.zip in My Drive'
os.environ['LANGSYS_WORK'] = str(HOME)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.chdir(HOME)
print('languages:', sorted(p.stem for p in (HOME / 'languages').glob('*.json')))

In [ ]:
#@title 3 · Install the tools (about 2 minutes)
%pip -q install piper-tts pandera soundfile diffusers transformers accelerate gradio
if FRENCH_VOICE:
    !apt-get -qq install -y espeak-ng > /dev/null
    %pip -q install kokoro
!ffmpeg -version | head -1

In [ ]:
#@title 4 · Language model for scene descriptions (only if chosen in step 1)
import time, requests, subprocess
if LANGUAGE_MODEL:
    if subprocess.run('which ollama', shell=True).returncode != 0:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        try:
            requests.get('http://localhost:11434/api/tags', timeout=2); break
        except Exception:
            time.sleep(1)
    !ollama pull llama3.1:8b 2>&1 | tail -1
    print('language model ready')
else:
    print('skipped: choose Simple scene descriptions on the page')

In [ ]:
#@title 5 · Publishing token (optional)
# To publish from the page, add a secret named NETLIFY_AUTH_TOKEN
# (key icon on the left, then Notebook access on). See docs/PRODUCTION.md.
import os
try:
    from google.colab import userdata
    os.environ['NETLIFY_AUTH_TOKEN'] = userdata.get('NETLIFY_AUTH_TOKEN')
    print('Netlify token loaded')
except Exception:
    print('no Netlify token: everything works except the Publish option')
try:
    os.environ['LANGSYS_PANEL_PASSWORD'] = userdata.get('PANEL_PASSWORD')
except Exception:
    pass

In [ ]:
#@title 6 · Open the production page
# Prints a link ending in gradio.live, and the user name and password.
!python bin/panel.py --share